# Serverless LLMs and Agentic AI with Modal – Lesson 5  
## Volumes: Persistent Storage Across Containers

In this lesson you’ll build a tiny **“Experiment Tracker”** for your serverless AI workflows.

Instead of writing “hello/bye” to a file, we’ll store **benchmark results** (JSON lines) in a **Modal Volume** so that:

- Results persist across runs (today, tomorrow, next week).
- Multiple Modal functions can read/write the same data.
- You learn the two key operational concepts:
  - **`commit()`** — make writes durable
  - **`reload()`** — fetch the latest committed data in containers that may be re-used

### What you’ll create
A script `lesson5_volumes_tracker.py` with three functions:

1. `log_run(...)`  
   Appends one benchmark record (a dict) to a JSONL file in a shared volume.

2. `tail_runs(reload=False, n=5)`  
   Reads the last *n* records. Demonstrates why `reload()` matters.

3. `leaderboard(reload=False)`  
   Produces a Markdown “leaderboard” table (fastest runs first).

At the end you’ll run a demo sequence that intentionally shows **stale reads** if you skip `reload()`.


In [ ]:
# =====================================
# Step 0 – Install and check Modal
# =====================================
!pip install modal --quiet
!which modal
!modal --version
print("✅ Modal installed.")

## Step 1 – Verify authentication



In [ ]:
# ============================
# Step 1– Configure Modal using the CLI (matches docs)
# ============================
# ⚠️ IMPORTANT:
# - Replace the placeholder strings with your real MODAL_TOKEN_ID and MODAL_TOKEN_SECRET.
# - Do NOT commit these values to GitHub or share them.
#
# This cell:
#   1. Stores your token via `modal token set`.
#   2. This writes the Modal config file (e.g. ~/.modal.toml) for you.

#TOKEN_ID = "YOUR_TOKEN_ID_HERE"        # <-- paste from Modal dashboard
#TOKEN_SECRET = "YOUR_TOKEN_SECRET_HERE"  # <-- paste from Modal dashboard


TOKEN_ID = ""        # <-- paste from Modal dashboard
TOKEN_SECRET = ""  # <-- paste from Modal dashboard



if "YOUR_TOKEN_ID_HERE" in TOKEN_ID or "YOUR_TOKEN_SECRET_HERE" in TOKEN_SECRET:
    raise ValueError("❌ Please set TOKEN_ID and TOKEN_SECRET before running this cell.")

# Call the Modal CLI to store the token
!modal token set --token-id $TOKEN_ID --token-secret $TOKEN_SECRET

print("✅ Token stored via `modal token set`. You should be authenticated now.")

## Step 2 – Write the Lesson 5 app

We’ll create a **named volume** so it’s easy to find in the Modal dashboard:

- Volume name: `lesson5-experiment-tracker`
- Mount point in containers: `/vol`

Inside that volume we store:
- `/vol/runs.jsonl` — one JSON dict per line

### Why JSONL?
It’s simple, append-friendly, and easy to inspect in the dashboard.


In [ ]:
%%writefile lesson5_volumes_tracker.py
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List

import modal

app = modal.App("lesson5-volumes-experiment-tracker")

# ------------------------------------------------------------
# Volume: persistent storage (shared across containers & runs)
# ------------------------------------------------------------
volume = modal.Volume.from_name("lesson5-experiment-tracker", create_if_missing=True)

# We'll mount the volume at /vol in every function that needs it.
VOL_DIR = Path("/vol")
RUNS_PATH = VOL_DIR / "runs.jsonl"


def _utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")


def _append_jsonl(path: Path, record: Dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _read_jsonl(path: Path) -> List[Dict]:
    if not path.exists():
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


# ------------------------------------------------------------
# Function 1: write a run result (append to JSONL)
# ------------------------------------------------------------
@app.function(volumes={"/vol": volume})
def log_run(model: str, device: str, items_per_sec: float, notes: str = "") -> Dict:
    """Append a new run record to the shared volume and commit it."""
    record = {
        "ts_utc": _utc_now(),
        "model": model,
        "device": device,
        "items_per_sec": float(items_per_sec),
        "notes": notes,
    }

    _append_jsonl(RUNS_PATH, record)

    # ✅ Persist changes so other containers/apps can see them.
    volume.commit()

    return {"status": "ok", "written_to": str(RUNS_PATH), "record": record}


# ------------------------------------------------------------
# Function 2: read last N runs (optionally reload volume)
# ------------------------------------------------------------
@app.function(volumes={"/vol": volume})
def tail_runs(reload: bool = False, n: int = 5) -> List[Dict]:
    """Read the last N run records.

    Why reload?
    - Modal may reuse a warm container for this function.
    - That container might have an older view of the volume.
    - `volume.reload()` fetches the latest committed state.
    """
    if reload:
        volume.reload()

    rows = _read_jsonl(RUNS_PATH)
    return rows[-n:]


# ------------------------------------------------------------
# Function 3: produce a Markdown leaderboard (portable string)
# ------------------------------------------------------------
@app.function(volumes={"/vol": volume})
def leaderboard(reload: bool = False, top_k: int = 10) -> str:
    """Return a Markdown leaderboard table (sorted by throughput)."""
    if reload:
        volume.reload()

    rows = _read_jsonl(RUNS_PATH)
    if not rows:
        return "# Leaderboard\n\nNo runs logged yet.\n"

    rows_sorted = sorted(rows, key=lambda r: r.get("items_per_sec", 0.0), reverse=True)[:top_k]

    lines = []
    lines.append("# Leaderboard (Top by items/sec)\n")
    lines.append("| Rank | items/sec | device | model | ts_utc | notes |\n|---:|---:|---|---|---|---|")
    for i, r in enumerate(rows_sorted, start=1):
        lines.append(
            f"\n| {i} | {r['items_per_sec']:.2f} | {r['device']} | {r['model']} | {r['ts_utc']} | {r.get('notes','')} |"
        )
    return "".join(lines) + "\n"


@app.local_entrypoint()
def lesson5_main():
    """Demonstration sequence.

    This intentionally shows why reload matters.
    """
    print("\n====================================")
    print("Lesson 5 – Volumes (Persistent Storage)")
    print("====================================\n")

    print("1) Log first run (commit happens inside log_run)\n")
    print(log_run.remote(model="miniLM", device="cpu", items_per_sec=120.5, notes="baseline"))

    print("\n2) Read tail WITHOUT reload (often OK on first read)\n")
    print(tail_runs.remote(reload=False, n=3))

    print("\n3) Log a second run (new commit)\n")
    print(log_run.remote(model="miniLM", device="gpu:A10G", items_per_sec=950.2, notes="gpu test"))

    print("\n4) Read tail again WITHOUT reload (may be STALE if container reused)\n")
    for i in range(3):
        print(f"  attempt {i+1} ->", tail_runs.remote(reload=False, n=3))

    print("\n5) Read tail WITH reload (should show latest committed data)\n")
    print(tail_runs.remote(reload=True, n=3))

    print("\n6) Leaderboard WITH reload\n")
    print(leaderboard.remote(reload=True, top_k=10))

    print("\n✅ Next: open the Modal dashboard → Storage → Volumes → lesson5-experiment-tracker")


## Step 3 – Run the demo

This will:
- create the volume (if missing)
- write two benchmark records
- demonstrate stale reads without `reload`
- show the leaderboard


In [ ]:
!modal run lesson5_volumes_tracker.py



```
# This is formatted as code
```

## Step 4 – Inspect the volume from CLI

Try these in a terminal:

```bash
modal volume list
modal volume ls lesson5-experiment-tracker
```

And you can shell into a container for debugging:
```bash
modal shell lesson5_volumes_tracker.py::tail_runs
```
Inside the container:
- `ls -la /vol`
- `tail -n 5 /vol/runs.jsonl`


In [ ]:
!modal volume list

In [ ]:
!modal volume ls lesson5-experiment-tracker

In [ ]:
!modal shell lesson5_volumes_tracker.py::tail_runs